<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_pinn_burgers_front_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 — A Front Carried by a Flow

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L9.1 · Laminar Flow**

Two streams of the same fluid meet. Below a slanted line the fluid moves
fast, towards the line; above it the fluid moves along the line and does not
cross it. So the fluid from below runs into the fluid above, and where they
meet the velocity changes from one value to the other in a thin band. That
band is a **front**: like the back of a queue of traffic, where fast cars
catch up with slow ones. Viscosity keeps it from becoming a sudden jump, and
the front itself creeps slowly upwards.

The equations are the **coupled Burgers equations**: the momentum equations
of a flow, Navier-Stokes, with the pressure left out. It is a model problem,
not a device, and it is chosen because it has an **exact solution**: every
error below is measured against the truth, not estimated. You solve it two
ways - finite differences on a grid, and a physics-informed network with two
outputs - and compare them on accuracy and on time. Then you make the fluid
less viscous, which makes the front thinner, and do it again.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the exact solution, and finite differences on a grid for an agreed 1e-03 | — |
| **5** | a PINN with two outputs, and the comparison | the two residuals, the loss |
| **6** | the Reynolds number raised from 20 to 500 | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

Two to four minutes on a CPU. Speeds are printed with at most two decimals;
errors as powers of ten.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex09.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

Look at a square window in the fluid, with $x$ to the right and $y$ upwards.
The velocity has two components: $u$ along $x$ and $v$ along $y$.

* **Below** the slanted line $y = x$ the fluid moves with $(u, v)$ =
  (0.50, 1.00): mostly upwards, towards the line.
* **Above** it the fluid moves with (0.75, 0.75): along the line, so it does
  not cross it.

The fluid from below runs into the fluid above. Between them the velocity
changes from the one value to the other in a thin band, the **front**, and the
front moves up by a quarter of a side while you watch.

Wanted: $u$ and $v$ at every point of the window over that time, and how thin
the front is.

## 2 · The data

Lengths are in units of the side $L$, speeds in units of a reference speed
$U$, time in units of $L/U$. In these units the fluid has one property left:

| | |
|---|---|
| region, window | the unit square, $0 \le t \le 1$ |
| Reynolds number | $\mathrm{Re} = UL/\nu$ = 20, so $\nu$ = 0.05 in scaled units |
| on the four edges, and at $t = 0$ | $u$ and $v$ given, from the exact solution |
| section 6 | the same at Re = 500, $\nu$ = 0.002 |

For water ($\nu$ = 1.0e-06 m²/s) in a 20 mm device, Re = 20 is a speed of
1 mm/s and Re = 500 is 25 mm/s.

## 3 · The physics

Each component is carried by the flow and smoothed by viscosity (L9.1's
*Coupled Burgers* slide):

$$u_t + u\,u_x + v\,u_y = \nu\,(u_{xx} + u_{yy}), \qquad
  v_t + u\,v_x + v\,v_y = \nu\,(v_{xx} + v_{yy})$$

Read the equation for $u$ term by term:

* $u_t$ - how fast the velocity changes at a fixed point;
* $u\,u_x + v\,u_y$ - the **convective** terms: the velocity is carried along
  by the flow itself. This is what steepens the front, as fast fluid runs into
  slow fluid;
* $\nu\,(u_{xx} + u_{yy})$ - **viscosity**, with $\nu$ the kinematic
  viscosity: it smooths the velocity out, and keeps the front from becoming a
  jump.

The front is where the two balance. The convective terms are nonlinear, and
they couple the two equations: $v$ appears in the equation for $u$, and $u$ in the equation for
$v$. These are the Navier-Stokes momentum equations with the pressure and the
incompressibility constraint left out: everything that makes a flow hard to
solve, except the constraint.

**The exact solution** is one front along the line $y = x + t/4$:

$$u = \tfrac34 - \frac{1}{4\,(1 + e^{s})}, \qquad v = \tfrac34 + \frac{1}{4\,(1 + e^{s})},
  \qquad s = \frac{(y - x) - t/4}{8\nu}$$

Below the line $s$ is large and negative, so $u$ = 0.50 and $v$ = 1.00; above
it $s$ is large and positive, so $u = v$ = 0.75; the front is where $s$ is
near zero. Two things follow before anything is solved. The front's width is $8\nu =
8/\mathrm{Re}$: 0.40 at Re = 20, and 0.016 at Re = 500. **The Reynolds number
is a length.** And $u + v = 3/2$ everywhere and always: a check that needs no
exact solution. The cell below prints the data and draws $u$ at the start and
at the end, with arrows for the velocity.

In [ ]:
pb.describe_problem()
g = np.linspace(0, 1, 101); GX, GY = np.meshgrid(g, g)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, t in zip(axes, (0.0, pb.T_END)):
    im = ax.contourf(GX, GY, pb.exact(GX, GY, t)[0], 21, cmap="viridis")
    ax.plot(g, g + t / 4, "w--", lw=1)                         # the front's line, y = x + t/4
    q = np.linspace(0.05, 0.95, 10); QX, QY = np.meshgrid(q, q)  # a coarse grid for the arrows
    QU, QV = pb.exact(QX, QY, t)
    ax.quiver(QX, QY, QU, QV, color="w", scale=12, width=0.006)  # the velocity (u, v): the two streams
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title(f"u at t = {t:.0f}, Re = {pb.RE:.0f}")
fig.colorbar(im, ax=axes, shrink=0.8, label="u")
plt.show()

**What you should see.** The data of section 2, a front 0.40 wide that moves
0.25 of a side, $u$ from 0.52 to 0.73 at the start; and two maps of $u$ with
the front's line, which has moved up by a quarter of a side in the second.
Below the line the arrows point steeply upwards, towards it; above it they
point along it. At Re = 20 the front is wide - it spans much of the window -
and in section 6 it becomes a thin line.

---

## 4 · The reference, and finite differences

**The reference** is the exact solution, `pb.exact(x, y, t, nu)`: nothing has
to be converged.

**Finite differences.** A square grid of $n \times n$ nodes, central
differences in space for the convective and the viscous terms, and a
two-stage step in time (Heun's method). The values on the four edges are the
given ones. The time step is the largest that is stable: a fifth of
$h^2/\nu$ for the viscosity, and of $h/U$ for the convection.

**The agreed accuracy is 1e-03** in $u$ and $v$, over the whole square and the
whole window: 0.4 % of the change of 0.25 across the front. The cell below
finds the coarsest grid that meets it.

In [ ]:
def fdm_burgers(n, nu):
    # n x n nodes; returns the nodes, u and v at t = 1, the worst error over the window and the number of steps
    h = 1.0 / (n - 1)
    g = np.linspace(0, 1, n); X, Y = np.meshgrid(g, g)
    steps = int(np.ceil(pb.T_END / min(0.2 * h * h / nu, 0.2 * h))); dt = pb.T_END / steps   # the stable step
    I = (slice(1, -1), slice(1, -1))                           # the interior nodes
    def rate(u, v):
        # du/dt and dv/dt at the interior nodes: minus the convection, plus the viscosity
        out = []
        for f in (u, v):
            fx = (f[1:-1, 2:] - f[1:-1, :-2]) / (2 * h); fy = (f[2:, 1:-1] - f[:-2, 1:-1]) / (2 * h)
            lap = (f[1:-1, 2:] + f[1:-1, :-2] + f[2:, 1:-1] + f[:-2, 1:-1] - 4 * f[I]) / h ** 2
            out.append(-u[I] * fx - v[I] * fy + nu * lap)
        return out
    def advance(u, v, du, dv, t):
        # the given values on the edges at time t, the stepped values inside
        un, vn = pb.exact(X, Y, t, nu)
        un[I] = u[I] + dt * du; vn[I] = v[I] + dt * dv
        return un, vn
    u, v = pb.exact(X, Y, 0.0, nu); worst = 0.0
    for k in range(steps):
        t = (k + 1) * dt
        du1, dv1 = rate(u, v)
        up, vp = advance(u, v, du1, dv1, t)                    # stage one: a full step with the rate at the start
        du2, dv2 = rate(up, vp)
        u, v = advance(u, v, 0.5 * (du1 + du2), 0.5 * (dv1 + dv2), t)   # stage two: the mean of the two rates
        ue, ve = pb.exact(X, Y, t, nu)
        worst = max(worst, np.abs(u - ue).max(), np.abs(v - ve).max())
    return X, Y, u, v, worst, steps

TARGET = 1e-3                                                 # the agreed accuracy, in units of U
for n in (6, 11, 21, 41):
    t0 = time.perf_counter(); X, Y, U, V, err, steps = fdm_burgers(n, pb.NU); sec = time.perf_counter() - t0
    print(f"  {n:3d} x {n:<3d} nodes, {steps:4d} steps: spacing {1 / (n - 1):.3f} against a front {pb.front_width():.2f} wide, "
          f"worst error {err:.2e}, {sec:.2f} s" + ("   <- meets 1e-03" if err <= TARGET else ""))
N_FDM = 6                                                     # the coarsest grid that meets it

**What you should see.** The error falling by four with every halving of the
spacing - 5.46e-04, 1.16e-04, 2.76e-05 and 6.85e-06 - which is second order. **A grid of
6 × 6 nodes already meets 1e-03**, in a few milliseconds: at Re = 20 the front
is 0.40 wide, twice the spacing of that grid, and a front that the grid
resolves is an easy problem.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(x, y, t)$ and returns **two** numbers, $u$ and $v$:
one network with two outputs, L9.1's *Shared Network*, because the two
components are coupled and should share their hidden layers. It is four
layers of 32 tanh neurons.

**The points.** 4000 collocation points spread over the square and the window,
where the two residuals are asked to vanish; and the points where the solution
is given - 20 on each edge at 12 instants, and 400 in the square at $t = 0$.
The given values are a **loss term**, with the weight $w_D$ = 10: the edges'
values change in time, so nothing is built in here.

**TODO 1** is the two residuals. Take the gradient of each output once -
its columns are the $x$, $y$ and $t$ derivatives - and use `d2` for the
viscous terms. **TODO 2** is the loss: the two mean squared residuals, and
$w_D$ times the misfit to the given values.

In [ ]:
pts = to_tensor(pb.sample_spacetime(4000), requires_grad=True)   # collocation points (x, y, t), differentiable
data_np = pb.sample_edges_and_start()                         # where the solution is given: the edges in time, and t = 0
data = to_tensor(data_np)                                     # the same points as a tensor
print(f"{len(pts)} collocation points, {len(data)} points with given values "
      f"({len(data) - 400} on the edges, 400 at the start)")

In [ ]:
def residuals(model, p, nu):
    out = model(p); u, v = out[:, 0:1], out[:, 1:2]           # the network's two outputs
    gu, gv = grad(u, p), grad(v, p)                           # first derivatives by autograd; columns x, y, t
    lap_u = d2(u, p, 0) + d2(u, p, 1)                         # second derivatives by autograd: u_xx + u_yy
    lap_v = d2(v, p, 0) + d2(v, p, 1)                         # and v_xx + v_yy
    res_u = gu[:, 2:3] + u * gu[:, 0:1] + v * gu[:, 1:2] - nu * lap_u   # the u equation; v enters it here
    res_v = gv[:, 2:3] + u * gv[:, 0:1] + v * gv[:, 1:2] - nu * lap_v   # the v equation; u enters it here
    return res_u, res_v

In [ ]:
W_D = 10.0                                                    # the weight of the given values
def make_loss(model, nu):
    u_d, v_d = pb.exact(data_np[:, 0], data_np[:, 1], data_np[:, 2], nu)   # the given values, computed once
    u_d, v_d = to_tensor(u_d.reshape(-1, 1)), to_tensor(v_d.reshape(-1, 1))   # as column tensors
    def loss_fn():
        res_u, res_v = residuals(model, pts, nu)              # the two residuals at the collocation points
        out = model(data)                                     # the network at the points with given values
        misfit = (out[:, 0:1] - u_d).pow(2).mean() + (out[:, 1:2] - v_d).pow(2).mean()   # mean squared misfit
        return res_u.pow(2).mean() + res_v.pow(2).mean() + W_D * misfit   # the composite loss
    return loss_fn

In [ ]:
def train_flow(nu):
    # one network for the given viscosity: returns it, and the seconds it took to train
    set_seed(88)                                              # the same start for every run
    model = MLP(n_in=3, n_out=2, n_hidden=32, n_layers=4).to(DEVICE)   # dense tanh network: x, y, t in; u, v out
    t0 = time.perf_counter()
    # Adam first, then L-BFGS: the Part 2 training recipe
    train_two_stage(model, make_loss(model, nu), adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
    return model, time.perf_counter() - t0

def score(model, nu):
    # the worst error in u and v, and the worst |u + v - 1.5|, on points spread over the square and laid across the front
    q = pb.check_points(nu)
    with torch.no_grad():  # no gradient tracking: evaluation only
        out = to_numpy(model(to_tensor(q)))                   # the network's u and v at the check points
    ue, ve = pb.exact(q[:, 0], q[:, 1], q[:, 2], nu)
    return max(np.abs(out[:, 0] - ue).max(), np.abs(out[:, 1] - ve).max()), np.abs(out[:, 0] + out[:, 1] - 1.5).max()

model, train_time = train_flow(pb.NU)                         # the network at Re = 20
describe(model, n_collocation=len(pts))                       # the parameter count and the collocation-to-parameter ratio
print(f"trained in {train_time:.0f} s")

In [ ]:
t0 = time.perf_counter(); X, Y, U, V, e_fdm, _ = fdm_burgers(N_FDM, pb.NU); t_fdm = time.perf_counter() - t0
e_pin, ident = score(model, pb.NU)
grid = np.c_[GX.ravel(), GY.ravel()]                          # 101 x 101 points for the network's field
def pinn_field(t, net=None):
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy((net or model)(to_tensor(np.c_[grid, np.full(len(grid), t)])))   # u, v on the grid at time t
t_pin = middle(lambda: [pinn_field(t) for t in np.linspace(0, pb.T_END, 21)], reps=3)

print(f"{'':28s}{'worst error':>13s}{'|u + v - 1.5|':>15s}{'one solution':>14s}{'training':>10s}")
print(f"  {'finite differences, 6 x 6':26s}{e_fdm:13.2e}{np.abs(U + V - 1.5).max():15.2e}{t_fdm:12.3f} s{'-':>10s}")
print(f"  {'PINN, 4 x 32, two outputs':26s}{e_pin:13.2e}{ident:15.2e}{t_pin:12.3f} s{train_time:8.0f} s")

def across_front(m=801):
    # points on the line x + y = 1, which crosses the front at right angles; at t = 1 the front is at x = 0.375
    x = np.linspace(0, 1, m); return x, 1 - x
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
xs, ys = across_front()
out = to_numpy(model(to_tensor(np.c_[xs, ys, np.full(len(xs), pb.T_END)])).detach())   # the network on that line
axes[0].plot(xs, pb.exact(xs, ys, pb.T_END)[0], "k", lw=2.5, label="exact")
axes[0].plot(xs, out[:, 0], ":", lw=2.5, color="tab:orange", label="PINN")
gx = np.linspace(0, 1, N_FDM)
axes[0].plot(gx, U[np.arange(N_FDM)[::-1], np.arange(N_FDM)], "s", color="tab:blue", label="finite differences, 6 x 6")
axes[0].set_xlabel("x, along the line x + y = 1"); axes[0].set_ylabel("u at t = 1"); axes[0].legend(frameon=False)
axes[1].semilogy(xs, np.abs(out[:, 0] - pb.exact(xs, ys, pb.T_END)[0]) + 1e-12, color="tab:orange", label="PINN")
axes[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 1e-03")
axes[1].set_xlabel("x, along the line x + y = 1"); axes[1].set_ylabel("error in u at t = 1"); axes[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex09.1_report.png"), dpi=150); plt.show()

**What you should see.** Both methods inside the agreed 1e-03: finite
differences at 5.46e-04 and the network at about 6.72e-04. The network keeps
$u + v$ within about 7e-04 of 3/2 without being told to; finite differences keep
it to rounding, because their scheme treats the two components alike.
Finite differences take a few milliseconds and no training; the network needs
about 40 s of training on a CPU. **At this Reynolds number the grid wins on
every count.**

---

## 6 · The Reynolds number, raised

Now Re = 500. The equations, the square and the window are the same; the
viscosity is 25 times smaller and the front is 0.016 wide instead of 0.40.
Both methods are run again with nothing changed but $\nu$: finite differences
on finer and finer grids, and the network of section 5 - the same size, the
same 4000 points - trained from the same start.

This is L9.1's *The Reynolds Number* slide as an experiment: the front's width
against the spacing of whatever is used to resolve it.

In [ ]:
nu_hi = pb.NU_HIGH                                            # Re = 500
print(f"Re = {pb.RE_HIGH:.0f}: the front is {pb.front_width(nu_hi):.3f} wide")
for n in (41, 81, 161, 321):
    t0 = time.perf_counter(); _, _, _, _, err, steps = fdm_burgers(n, nu_hi); sec = time.perf_counter() - t0
    print(f"  {n:3d} x {n:<3d} nodes, {steps:4d} steps: spacing {1 / (n - 1):.4f}, "
          f"{pb.front_width(nu_hi) * (n - 1):.1f} nodes across the front, worst error {err:.2e}, {sec:.2f} s"
          + ("   <- meets 1e-03" if err <= TARGET else ""))
    e_fdm_hi, t_fdm_hi = err, sec
model_hi, train_hi = train_flow(nu_hi)                        # the same network and points, the new viscosity
e_pin_hi, ident_hi = score(model_hi, nu_hi)
print(f"  PINN, 4 x 32, the same 4000 points: worst error {e_pin_hi:.2e}, |u + v - 1.5| {ident_hi:.2e}, trained in {train_hi:.0f} s")

k = np.abs(xs - 0.375) < 0.05                                 # a close-up of the front on the line x + y = 1
out = to_numpy(model_hi(to_tensor(np.c_[xs, ys, np.full(len(xs), pb.T_END)])).detach())   # the network on that line
plt.figure(figsize=(5.5, 3.4))
plt.plot(xs[k], pb.exact(xs[k], ys[k], pb.T_END, nu_hi)[0], "k", lw=2.5, label="exact")
plt.plot(xs[k], out[k, 0], ":", lw=2.5, color="tab:orange", label="PINN")
plt.xlabel("x, along the line x + y = 1"); plt.ylabel("u at t = 1"); plt.title("Re = 500")
plt.legend(frameon=False); plt.tight_layout(); plt.show()

**What you should see.** Finite differences now need **321 × 321 nodes** -
about five nodes across the front - to come under 1e-03, and about ten seconds in place
of milliseconds: 2.51e-02, 6.06e-03, 1.33e-03 and 3.08e-04. The network, with the same
4000 points, is at about 9.68e-04 after about a minute of training: its error
has hardly moved.

**Read that result with care.** 4000 points in the square and the window are
about 0.06 apart, four times the front's width, and L9.1's *Reynolds Ceiling*
says a sample like that cannot resolve the front. Here it does, for a reason
that belongs to this problem: the exact solution is a tanh of a straight line
in $(x, y, t)$, which is exactly what **one tanh neuron** computes. The
network does not have to resolve the front with points; it has to find one
steep neuron, and the given values at the start and on the edges tell it
where. A boundary layer on a curved wall or a wake is not one neuron - that
is Ex_09.2.

---

## 7 · What the notebook says

* **At Re = 20 the grid wins on every count.** Finite differences meet the
  agreed accuracy on 6 × 6 nodes in milliseconds. The network meets it too,
  after 40 s of training.
* **The Reynolds number is a length.** The front is 8/Re wide. A grid needs
  about five nodes across it, so the grid's cost grows with the Reynolds
  number: 6 nodes a side at Re = 20, 321 at Re = 500, and each halving of the
  spacing costs eight times the work.
* **The network's error did not grow from Re = 20 to Re = 500** - on this
  problem, whose front is a single tanh. That is a property of the solution,
  not of the method, and it is the reason to test a method on more than one
  problem before trusting it.
* **$u + v = 3/2$ is a check that needs no exact solution.** The network was
  never told it and keeps it to about 1e-03. A check like that - an identity,
  a balance, a symmetry - is what remains when there is no formula to compare
  with.
* **One network, two outputs.** The two components share their hidden layers;
  the coupling of the equations is in the residuals you wrote, where each
  component multiplies the other's gradient.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex09.1_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L9.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Which terms of your residuals couple the two equations, and what do the coupled Burgers equations leave out of the equations of a real flow? (-> L9.1 Q1)": """
""",
    "What length did the Reynolds number set in this problem, and what did raising it from 20 to 500 cost finite differences? (-> L9.1 Q2)": """
""",
    "The network has two outputs and was trained on the same 4000 points at both Reynolds numbers. Why did its error hardly change here, and why should you not expect that of a boundary layer or a wake? (-> L9.1 Q3)": """
""",
    "Which check in this notebook needs no exact solution, what did it give for the network, and what kind of error would it not catch? (-> L9.1 Q4)": """
""",
    "To conclude: which method would you hand to a colleague for this problem at Re = 20, and which at Re = 500, and what would you tell them about how far the network's result carries to other flows? (-> L9.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| Re | method | worst error | abs(u + v - 1.5) | one solution | training |", "|---|---|---|---|---|---|",
            f"| 20 | finite differences, 6 x 6 | {e_fdm:.2e} | {np.abs(U + V - 1.5).max():.2e} | {t_fdm:.3f} s | - |",
            f"| 20 | PINN, 4 x 32, two outputs | {e_pin:.2e} | {ident:.2e} | {t_pin:.3f} s | {train_time:.0f} s |",
            f"| 500 | finite differences, 321 x 321 | {e_fdm_hi:.2e} | - | {t_fdm_hi:.2f} s | - |",
            f"| 500 | PINN, the same network and points | {e_pin_hi:.2e} | {ident_hi:.2e} | - | {train_hi:.0f} s |"]
    out = ["# Ex_09.1 — A Front Carried by a Flow", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex09.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex09.1_report.md into Ex09.1_report.pdf, with any figure
# saved as Ex09.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex09.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex09.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP9.1A · Two fronts that meet

**The problem.** The flow of this set had one front and an exact solution. Here two streams run
into each other: they steepen into fronts along $x = 1/2$ and $y = 1/2$, and
the fronts cross at the centre. No formula gives the result - the reference
is a numerical one, as it will be for every real flow.

| | |
|---|---|
| equations | the coupled Burgers equations of this set on the unit square, $0 \le t \le 1$ |
| start | $u_0 = \sin(2\pi x)\sin(\pi y)$, $v_0 = \sin(\pi x)\sin(2\pi y)$ |
| walls | $u = v = 0$ |
| Reynolds number | 100 and 200 ($\nu$ = 0.01 and 0.005) |

**Why a plain PINN is not enough.** The notebook's network kept its accuracy as the front sharpened, because that
front is a single tanh and is there from the start. Here the fronts form
during the run, and where they cross the gradients are steepest: nothing
tells the network where to put a steep neuron, and a smooth, plausible, wrong
field with a healthy loss is what to expect first.

**What you build.** A PINN for $u, v$ that resolves the fronts: points that follow them, training
in time order, or features that let the network represent a sharp change. You
choose, and you show at which Reynolds number each choice stops working.

**The ground truth you get.** `MP9.1A_truth_Re100.npz` and `MP9.1A_truth_Re200.npz`, from
`tools/miniprojects/ex091_truth.py`: finite differences on a 257 × 257 grid,
second-order upwind for the convection and central for the diffusion, a
third-order Runge-Kutta step; $u$ and $v$ at $t$ = 0.25, 0.5 and 1.

**Worked example.** 

| grid | Re 100: steepest $\partial u/\partial x$ at $t$ = 1 | Re 200 | run, Re 100 |
|---|---|---|---|
| 64 × 64 | 4.48 | 2.98 | 0.1 s |
| 128 × 128 | 4.60 | 9.99 | 2.2 s |
| 256 × 256 | 4.63 | 10.24 | 29.0 s |

From 128 to 256 the field changes by at most 0.0007 at Re 100 and 0.0049 at
Re 200. At Re 200 the coarsest grid smears the front to a third of its true
steepness and still looks smooth - the failure this set warned about, made by
a classical method.

**Requirements.**

* $u$ and $v$ within **0.02** everywhere at Re 100, and within **0.05** at Re 200, at the three times.
* The steepest gradient at $t$ = 1 within **10 %** at both Reynolds numbers.
* The highest Reynolds number at which you meet both, with the evidence.

**What you hand in.** The error against the ground truth at each Reynolds number and time, where your
network puts the fronts against where they are, and the highest Reynolds
number at which you would report your own result.

### MP9.1B · What is the viscosity?

**The problem.** Every run of this set was told $\nu$. Here a few velocity probes record the
flow, and $\nu$ - and so the Reynolds number - is unknown. From the probes
alone, find the viscosity and the whole flow.

| | |
|---|---|
| flow | the exact solution of this set, its front on $y = x + t/4$, width $8\nu$ |
| records | $u$ and $v$ at 51 instants from 0 to 1, noise 1e-03 |
| case 1 | 6 probes near the front's path, $\nu$ = 0.02 (Re 50) |
| case 2 | the same probes, $\nu$ = 0.005 (Re 200) |
| case 3 | 3 probes in the far corners, $\nu$ = 0.005 |

**Why a plain PINN is not enough.** The viscosity sets the width of the front, so a probe can only see it if the
front passes it. Far from the front a probe says almost nothing about $\nu$ -
and a network will still return a number.

**What you build.** An inverse PINN: the network for $u, v$ and a trainable $\nu$ (as $\log\nu$),
fitted to the coupled Burgers equations and the probe records. You report
$\nu$ with an honest uncertainty.

**The ground truth you get.** `MP9.1B_readings.npz` to fit and `MP9.1B_truth.npz` with the hidden
viscosities, from the same script: the exact solution, sampled.

**Worked example.** A 10 % change of $\nu$ moves the readings of case 1 by up to 5.3e-03 -
five times the noise - and those of case 2 by up to 5.3e-03 too. In case 3
it moves them by at most 6.6e-07: a thousandth of the noise. Case 3 cannot
be solved, and saying so is the answer.

**Requirements.**

* Cases 1 and 2: $\nu$ within **5 %**, and $u, v$ within **0.01** everywhere.
* Case 3: show that $\nu$ cannot be recovered, and give the range your data allows.

**What you hand in.** The recovered viscosity with its uncertainty in each case, the flow's error, and
the probe placement that fails, and why.